## Sentence Splitting

In [1]:
from cleaning_helpers import *
import pandas as pd
import re

## Paragraph Formation

In [2]:
clean_file("C:/Users/Kamal/OneDrive/Desktop/PDS/corpus_txt/Divan_Prak10.txt", "divan_prak10_cleaned.txt")


Cleaned file saved as: divan_prak10_cleaned.txt


## Sentence Splitting

In [ ]:
import re
import pandas as pd

def is_all_caps_armenian(line: str) -> bool:
    armenian_letters = re.findall(r'[Ա-Ֆա-ֆ]', line)
    if not armenian_letters:
        return False
    upper_count = sum(1 for ch in armenian_letters if ch.isupper())
    return upper_count / len(armenian_letters) > 0.8

def split_paragraph_custom(paragraph: str) -> list:
    n = len(paragraph)
    start = 0
    i = 0
    sentences = []
    while i < n:
        if paragraph[i] == ':':
            end = i + 1
            sentences.append(paragraph[start:end].strip())
            start = end
            i = end
            continue
        if paragraph[i:i+2] == '..':
            end = i + 2
            sentences.append(paragraph[start:end].strip())
            start = end
            i = end
            continue
        if paragraph[i] == '։':
            j = i + 1
            while j < n and paragraph[j].isspace():
                j += 1
            boundary = False
            if j >= n:
                boundary = True
            else:
                if paragraph[j].isupper():
                    boundary = True
            if boundary:
                end = i + 1
                sentences.append(paragraph[start:end].strip())
                start = end
                i = end
                continue
        i += 1
    if start < n:
        sentences.append(paragraph[start:].strip())
    return [s for s in sentences if s and not is_all_caps_armenian(s)]

def process_txt_file(filename: str) -> pd.DataFrame:
    with open(filename, "r", encoding="utf-8") as f:
        content = f.read()
    pages = re.split(r'--- Page (\d+) ---', content)
    rows = []
    for i in range(1, len(pages), 2):
        page_num = int(pages[i])
        page_text = pages[i+1].strip()
        paragraphs = [p for p in page_text.split("\n\n") if p.strip()]
        for p in paragraphs:
            sents = split_paragraph_custom(p)
            rows.extend((page_num, s) for s in sents)
    return pd.DataFrame(rows, columns=["page", "sentence"])


# Example usage
df = process_txt_file("divan_prak10_cleaned.txt")
print(df.head(20))


    page                                           sentence
0      1                              Ե 2 - ա -- ---- ----Բ
1      1                                                  4
2      1                                                  "
3      1                                                0 4
4      1                                               23 "
5      1                                                # 4
6      6  ########### ####### ######\n####### ##########...
7      6                         #### 3\n######### ########
8      6                                            8######
9      6                                                  X
10     6                                            #B### X
11     6            ######\n############ ############\n2017
12     7                                             ❇##797
13     7                                               2017
14     8  Տպագրվում է Հ3 ԳԱՍ հնագիտության և ազգագրության...
15     8  Խմբագրական խորհուրդ՝ Գր.Ս. Գրի